<a href="https://colab.research.google.com/github/AlexZhuo29/QCC_STEP_Research_2025_2026/blob/main/STEP_Research_Code_2025_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
## Libraries needed
import pandas as pd
import numpy as np

import yfinance as yf
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
import csv
#import requests

## 5 year time frame
START = "2010-1-1"
END = "2015-12-31"

## creating lag data (previous days)
def lag_func(dataframe, name, lag_day):
    ## creates a dataframe with just the Date and name column and assign it to df_lag
    df_lag = dataframe[['Date', name]].copy()

    ## start from day 1 to the lag_day provided b/c lag_day+1 is excluded
    for i in range(1, lag_day + 1):
        ## creates a new column and assign value into the column
        ## df_lag[name].shift(i) means to shift the previous value down
        df_lag[f'lag_{i}'] = df_lag[name].shift(i)

    ## removes empty spaces
    df_lag.dropna(inplace=True)

    ## return finished dataframe
    return df_lag

## process data and fit into dataframe
def process_func(name, symbol):
    ## create new empty dataframe
    df = pd.DataFrame()

    ## fill the dataframe with stock prices
    df[name] = yf.Ticker(symbol).history(start=START, end=END).Close

    ## add index
    df.reset_index(inplace=True)

    ## removes time
    df['Date'] = [i.date() for i in df.Date]

    ## adds a column for the stock price going up or down
    df['Up/Down'] = (df[name].shift(-1) > df[name]).astype(int)

    ## return dataframe
    return df

## Data Preparation
def prepare_func(name, symbol, lag_day):
  ## create a processed dataframe
  df = process_func(name, symbol)

  ## create a dataframe with the lag data
  df_lag = lag_func(df, name, lag_day)
  df_lag.head().round(4)

  ## update the lag dataframe's Up/Down with processed dataframe's Up/Down values at the same indices
  df_lag['Up/Down'] = df['Up/Down'].loc[df_lag.index]

  ## drop the empty cells
  df_lag.dropna(inplace=True)

  ## return dataframe
  return df_lag

## define input and output
def defineXY_func(dataframe): ## x is input and y is output
  ## lag data is the input
  x = dataframe[[f'lag_{i}' for i in range(1, 6)]]
  ## output is whether the stock went up or down
  y = dataframe['Up/Down']

  ## shape the number of x and y
  print(x.shape)
  print(y.shape)

  ## return input and output
  return x,y

## split data for training and testing
def splitData_func(x,y):
  ## initialize the scaler tool which processes the data so it will be compatiable with KNN
  scaler = StandardScaler()

  ## scaler figures out scaling parameters of x and apply it to the input
  x_scaled = scaler.fit_transform(x)

  ## splitting data into 80 / 20. 80% of the dataset are used for training and 20% used for testing
  split = int(len(x_scaled) * 0.8)

  ## assign front 80% to training data and remaining 20% to testing data
  x_train, x_test = x_scaled[:split], x_scaled[split:]
  y_train, y_test = y.iloc[:split], y.iloc[split:]

  ## return the assigned dataset
  return x_train, x_test, y_train, y_test

In [2]:
## companies that will be tested
company_name = ['Google', 'Oracle', 'JP Morgan', 'Johnson and Johnson', 'Delta', 'Costco']
company = ['GOOGL', 'ORCL', 'JPM', 'JNJ', 'DAL', 'COST']

## stored values for each individual companies
df = []
x_values = []
y_valies = []
x_train = []
x_test = []
y_train = []
y_test = []


## store dataframe that will contain the model scores
result_df = []

## split the testing and training for all companies being tested
def company_func(index):
  ## prepare the data
  df_company = prepare_func(company_name[index], company[index], 5)

  ## define the input and output
  x_company, y_company = defineXY_func(df_company)

  ## split the training and testing data
  x_company_train, x_company_test, y_company_train, y_company_test = splitData_func(x_company, y_company)

  ## save result to lists for later user
  df.append(df_company)
  x_values.append(x_company)
  y_valies.append(y_company)
  x_train.append(x_company_train)
  x_test.append(x_company_test)
  y_train.append(y_company_train)
  y_test.append(y_company_test)

  ## create empty dataframe for each company
  result_df.append(pd.DataFrame())

  ## verify which company's data was prepared
  print(company_name[index])



In [7]:
## library for KNN Classifier
from sklearn.neighbors import KNeighborsClassifier

## Argument for each parameter
KNN_NumNeighbors = range(1, 50, 3)
KNN_Weights = [None, 'uniform', 'distance']
KNN_Algorithm = ['auto', 'ball_tree', 'kd_tree', 'brute']
KNN_LeafSize = range(1, 50, 3)
KNN_Power = np.arange(1, 50, 3)

## Column Label
column_label = ['N_neighbors', 'Weights', 'Algorithm', 'Leaf Size', 'Power', 'Train Score', 'Test Score']

## KNN
def KNN_func(X_train, X_test, Y_train, Y_test, dataframe):
  ## create columns with the label and will have empty values
  dataframe[column_label] = np.nan

  ## for loop that checks every single combination of given parameters
  for NumNeighbors in KNN_NumNeighbors:
    for Weights in KNN_Weights:
      for Algorithm in KNN_Algorithm:
        for LeafSize in KNN_LeafSize:
          for Power in KNN_Power:

            ## declare and initialize the classical machine learning model
            knn = KNeighborsClassifier(n_neighbors = NumNeighbors, weights = Weights, algorithm = Algorithm, leaf_size = LeafSize, p = Power)

            ## train the model with the training input and output
            knn.fit(X_train, Y_train)

            ## prediction
            knn.predict(X_train[:5])
            knn.predict(X_test[:5])

            ## score the prediction
            trainScore = knn.score(X_train, Y_train)
            testScore = knn.score(X_test, Y_test)

            ## update dataframe
            dataframe.loc[len(dataframe)] = [NumNeighbors, Weights, Algorithm, LeafSize, Power, trainScore, testScore]

In [12]:
## loop that splits the dataset for all the companies being tested
for i in range(len(company)):
  company_func(i)

(1504, 5)
(1504,)
Google
(1504, 5)
(1504,)
Oracle
(1504, 5)
(1504,)
JP Morgan
(1504, 5)
(1504,)
Johnson and Johnson
(1504, 5)
(1504,)
Delta
(1504, 5)
(1504,)
Costco


In [ ]:
## loop that trains the model and check the prediction made
for i in range(1,len(company_name)):
  KNN_func(x_train[i],x_test[i],y_train[i],y_test[i],result_df[i])
  print(company[i])

Everything below this point is for exporting the compiled results

In [14]:
display(result_df[1])

,N_neighbors,Weights,Algorithm,Leaf Size,Power,Train Score,Test Score
0,1,None,auto,1,1,1.000000,0.491694
1,1,None,auto,1,4,1.000000,0.491694
2,1,None,auto,1,7,1.000000,0.481728
3,1,None,auto,1,10,1.000000,0.485050
4,1,None,auto,1,13,1.000000,0.481728
...,...,...,...,...,...,...,...
5706,4,uniform,brute,37,34,0.697423,0.518272
5707,4,uniform,brute,37,37,0.697423,0.521595
5708,4,uniform,brute,37,40,0.697423,0.524917
5709,4,uniform,brute,37,43,0.696592,0.524917


In [17]:
result_df[1].round(2).sort_values('Test Score', ascending=False).to_csv('/content/drive/My Drive/KNNClassGoogle.csv', index=False)

In [ ]:
## list that stores the list of parameters and the scores
df_rounded = []

## sort the values by test score and round the values
for i in range(len(company)):
  df1 = result_df[i].round(2).sort_values('Test Score', ascending=False)
  df_rounded.append(df1)

In [16]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
## save to google drive
for i in range(len(company)):
  df_rounded[i].to_csv(f'/content/drive/My Drive/KNNClass{i}.csv', index=False)